# Structure Track: Workflow to Submit Structure Predictions on the OpenADMET CYP Blind Challenge
### focus is on using OpenFold3 as a co-folding tool
### output from this is put through a variety of selection methods (md, knowledge based, etc)
### Several cells borrowed outright or adapted from the excellent tutorial provided by OpenADMET
### Otherwise - created by Byron DeLaBarre / The Consulting Biochemist Sept 2026
### Claude Opus 5.5 was used to generate and modify code where appropriate
### Code also generated manually when necessary

In [ ]:
import sys
import warnings

from pathlib import Path
import zipfile

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import time, requests

from tqdm.auto import tqdm
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw, Descriptors, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import AllChem, DataStructs

from matplotlib.offsetbox import OffsetImage, AnnotationBbox

from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.inchi import MolToInchiKey

from pymol import cmd

import json
import shutil

import biotite.structure.io as strucio

import os, glob
import gemmi
from posebusters import PoseBusters

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

from gradio_client import Client, handle_file
from huggingface_hub import get_token

tqdm.pandas()
sns.set_style("whitegrid")
sns.set_context("notebook")

In [ ]:
print (os.getcwd())

In [ ]:
def rewrite_ligand_resname_pdb(
    input_structure: Path,
    output_pdb: Path,
    old_resname: str = "LIG1",
    new_resname: str = "LIG",
):
    """Read a structure file and write a PDB with ligand residue name normalized."""
    if strucio is None:
        raise ImportError("biotite is required for residue renaming. Install with: pip install biotite")

    # Generic load handles CIF/PDB based on extension
    stack = strucio.load_structure(str(input_structure))

    if hasattr(stack, "res_name"):
        mask = stack.res_name == old_resname
        if np.any(mask):
            stack.res_name[mask] = new_resname

    strucio.save_structure(str(output_pdb), stack)
    return output_pdb

## 1. Loading the data

Load the blinded ligand list for the structure track directly from Hugging Face.

In [ ]:
test_structure = pd.read_csv("hf://datasets/openadmet/cyp-challenge-train-test/cyp-challenge-TEST-BLINDED_structures.csv")
test_structure.head()

## 2. Preparing the data for prediction

For the structure track, the main input is the ligand SMILES list so preprocessing is minimal:

- Keep molecule identifiers and SMILES.
- Parse SMILES into RDKit molecules.
- Define the protein context for the structure-generation pipeline.

For this tutorial, we load the CYP3A4 protein sequence from a local FASTA file stored in `inputs/CYP3A4_protein_sequence.fasta`.

You will be predicting 20 structures in total.

In [ ]:
required_cols = ["Molecule_Name", "SMILES"]
missing_cols = [col for col in required_cols if col not in test_structure.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in test_structure: {missing_cols}")

structure_df = test_structure[required_cols].copy().rename(columns={"Molecule_Name": "id"})
structure_df["mol"] = structure_df["SMILES"].apply(Chem.MolFromSmiles)

invalid_smiles = structure_df["mol"].isna().sum()
print(f"Total ligands: {len(structure_df)}")
print(f"Invalid SMILES: {invalid_smiles}")
structure_df.head()

In [ ]:
structure_df[["id", "SMILES"]].to_csv("cyp_challenge_compound.csv", index=False)
print("Saved: cyp_challenge_compound.csv")

## Using an excel file (pulled from unitprot) summarize all of the relevant files in the PDB
### (should only have to run this once - refer to the output csv file for future efforts)

In [ ]:

# ---- config ----
XLSX       = "cyp3a_PDB_2026.xlsx"   # adjust path
OUT_CSV    = "cyp3a4_pdb_summary.csv"
BATCH      = 25          # IDs per GraphQL request
HEME_CODES = {"HEM"}     # flagged separately; CYP3A4 always has heme
# ----------------

COMMON_ADDITIVES = {
    "HEM","HOH","GOL","EDO","PEG","PGE","PG4","P6G","MPD","SO4","PO4","CL","NA","MG",
    "K","ZN","CA","FE","ACT","FMT","DMS","TRS","MES","EPE","IMD","BME","IPA","NAG","BOG",
    "CPS","LDA","C8E","OLC","P33","1PE","2PE","TLA","CIT","BCT","NO3","IOD","BR","F",
}

# --- read the sheet (no header row) ---
df = pd.read_excel(XLSX, header=None)
df.columns = ["blank", "pdb_id", "method", "resolution", "chains", "residue_range"]
df = df.drop(columns=["blank"])
df["pdb_id"] = df["pdb_id"].astype(str).str.strip().str.upper()
ids = df["pdb_id"].tolist()
print(f"{len(ids)} PDB IDs from spreadsheet")

# --- GraphQL: titles + nonpolymer entity chem comps in batched calls ---
GQL = """
query($ids:[String!]!){
  entries(entry_ids:$ids){
    rcsb_id
    struct { title }
    rcsb_entry_info { resolution_combined }
    rcsb_accession_info { initial_release_date }
    nonpolymer_entities {
      nonpolymer_comp {
        chem_comp { id name formula_weight }
        rcsb_chem_comp_descriptor { SMILES SMILES_stereo }
      }
    }
  }
}
"""

records = {}
for i in range(0, len(ids), BATCH):
    chunk = ids[i:i+BATCH]
    r = requests.post("https://data.rcsb.org/graphql",
                      json={"query": GQL, "variables": {"ids": chunk}}, timeout=60)
    r.raise_for_status()
    data = r.json()
    if "errors" in data:
        print("GraphQL errors:", data["errors"][:2])
    for e in (data.get("data", {}).get("entries") or []):
        pid = e["rcsb_id"].upper()
        ligs = []
        for ne in (e.get("nonpolymer_entities") or []):
            comp = (ne.get("nonpolymer_comp") or {})
            cc   = comp.get("chem_comp") or {}
            desc = comp.get("rcsb_chem_comp_descriptor") or {}
            code = cc.get("id")
            if not code:
                continue
            ligs.append({
                "code": code,
                "name": cc.get("name"),
                "mw":   cc.get("formula_weight"),
                "smiles": desc.get("SMILES_stereo") or desc.get("SMILES"),
            })
        records[pid] = {
            "title": (e.get("struct") or {}).get("title"),
            "release_date": (e.get("rcsb_accession_info") or {}).get("initial_release_date"),
            "ligands": ligs,
        }
    print(f"  fetched {min(i+BATCH, len(ids))}/{len(ids)}")
    time.sleep(0.2)

# --- assemble summary: one row per PDB entry ---
rows = []
for _, r in df.iterrows():
    pid = r["pdb_id"]
    rec = records.get(pid, {})
    ligs = rec.get("ligands", [])
    drug = [l for l in ligs if l["code"] not in COMMON_ADDITIVES]   # candidate substrate/inhibitor
    heme = [l for l in ligs if l["code"] in HEME_CODES]
    rows.append({
        "pdb_id": pid,
        "method": r["method"],
        "resolution": r["resolution"],
        "chains": r["chains"],
        "residue_range": r["residue_range"],
        "title": rec.get("title"),
        "release_date": (rec.get("release_date") or "")[:10],
        "n_ligands_total": len(ligs),
        "has_heme": bool(heme),
        "n_drug_ligands": len(drug),
        "ligand_codes": ";".join(l["code"] for l in drug),
        "ligand_names": " | ".join(str(l["name"]) for l in drug),
        "ligand_smiles": " | ".join(str(l["smiles"]) for l in drug),
        "ligand_mw": ";".join(f"{l['mw']:.1f}" if l["mw"] else "" for l in drug),
        "all_het_codes": ";".join(l["code"] for l in ligs),
    })

summary = pd.DataFrame(rows)
summary.to_csv(OUT_CSV, index=False)

print(f"\n{len(summary)} entries -> {OUT_CSV}")
print(f"missing from API: {summary['title'].isna().sum()}")
print(f"with heme: {summary['has_heme'].sum()}")
print(f"with >=1 drug-like ligand: {(summary['n_drug_ligands']>0).sum()}")
print(f"apo (no drug ligand): {(summary['n_drug_ligands']==0).sum()}")
summary.head(10)

## Review entries in the PDB first
#

In [ ]:
# Download all files into a folder (should only have to run this once)


# ---- config ----
SUMMARY_CSV = "cyp3a4_pdb_summary.csv"      # from the previous cell
OUT_DIR     = "./cyp3a4_structures"
FMT         = "cif"        # "cif" (recommended) or "pdb"
ASSEMBLY    = False        # True = biological assembly 1; False = asymmetric unit
OVERWRITE   = False
# ----------------

os.makedirs(OUT_DIR, exist_ok=True)
ids = pd.read_csv(SUMMARY_CSV)["pdb_id"].astype(str).str.strip().str.upper().tolist()
print(f"{len(ids)} structures to fetch -> {OUT_DIR}/")

def url_for(pid):
    p = pid.lower()
    if ASSEMBLY:
        return (f"https://files.rcsb.org/download/{p}-assembly1.cif" if FMT == "cif"
                else f"https://files.rcsb.org/download/{p}.pdb1")
    return f"https://files.rcsb.org/download/{p}.{FMT}"

ok, skipped, failed = 0, 0, []
for i, pid in enumerate(ids, 1):
    suffix = f"-assembly1.{FMT}" if ASSEMBLY else f".{FMT}"
    dest = os.path.join(OUT_DIR, f"{pid}{suffix}")
    if os.path.exists(dest) and os.path.getsize(dest) > 0 and not OVERWRITE:
        skipped += 1; continue
    try:
        r = requests.get(url_for(pid), timeout=60)
        r.raise_for_status()
        with open(dest, "wb") as fh:
            fh.write(r.content)
        ok += 1
        if i % 20 == 0 or i == len(ids):
            print(f"  {i}/{len(ids)} (downloaded {ok}, skipped {skipped})")
    except Exception as e:
        failed.append((pid, str(e)))
        print(f"  {pid}: FAILED {e}")
    time.sleep(0.15)          # be polite to RCSB

print(f"\ndownloaded {ok} | already present {skipped} | failed {len(failed)}")
if failed:
    for pid, why in failed: print(f"  {pid}: {why}")

# quick inventory
sizes = [(f, os.path.getsize(os.path.join(OUT_DIR, f))) for f in sorted(os.listdir(OUT_DIR))]
print(f"\n{len(sizes)} files on disk, total {sum(s for _, s in sizes)/1e6:.1f} MB")

In [ ]:
# Create pymol session for aligning cyp3a4 structures
# cyp3a4_align.pml  — run with: pymol cyp3a4_align.pml


KEEP_CHAIN = None     # None = first polymer chain per entry; or force e.g. "A"

files = sorted(glob.glob("./cyp3a4_structures/*.cif"))
ref, kept, skipped = None, [], []

for f in files:
    name = os.path.basename(f).split(".")[0].split("-")[0]
    tmp = f"tmp_{name}"
    cmd.load(f, tmp)

    # pick the chain: forced, or the first polymer chain present
    chains = cmd.get_chains(f"{tmp} and polymer")
    if not chains:
        cmd.delete(tmp); skipped.append(name); continue
    ch = KEEP_CHAIN if (KEEP_CHAIN and KEEP_CHAIN in chains) else sorted(chains)[0]

    # keep that chain only: protein + its heme/ligands (same auth chain)
    cmd.create(name, f"{tmp} and chain {ch} and not solvent")
    cmd.delete(tmp)

    if cmd.count_atoms(name) == 0:
        cmd.delete(name); skipped.append(name); continue

    if ref is None:
        ref = name
    else:
        cmd.super(f"{name} and polymer", f"{ref} and polymer")   # moves whole object
    kept.append((name, ch, len(chains)))

cmd.hide("everything")
cmd.show("cartoon", "polymer")
cmd.show("sticks", "organic")
cmd.color("grey80", "polymer")
cmd.util.cbay("organic")
cmd.orient(ref)

multi = [(n, c, k) for n, c, k in kept if k > 1]
print(f"aligned {len(kept)} structures onto {ref}")
print(f"multi-copy entries reduced to one chain: {len(multi)}")
for n, c, k in multi[:15]:
    print(f"   {n}: {k} chains -> kept {c}")
if skipped:
    print(f"skipped (no polymer): {skipped}")

OUT_SESSION = "./cyp3a4_aligned.pse"
OUT_ALIGNED = "./cyp3a4_aligned"          # per-structure aligned coordinate files

# 1) save the whole session (all objects, colors, view) — reopen with: pymol cyp3a4_aligned.pse
cmd.save(OUT_SESSION)
print(f"session -> {OUT_SESSION}")

# 2) save each aligned single-chain structure as its own file
#import os
#os.makedirs(OUT_ALIGNED, exist_ok=True)
#for n, _, _ in kept:
#    cmd.save(os.path.join(OUT_ALIGNED, f"{n}_aligned.pdb"), n)
#print(f"{len(kept)} aligned structures -> {OUT_ALIGNED}/")

In [ ]:
# compare PDB entries to challenge entries


RDLogger.DisableLog("rdApp.*")

# ---- config ----
SUMMARY_CSV = "cyp3a4_pdb_summary.csv"
CHALLENGE_SMILES_COL = "SMILES"
CHALLENGE_ID_COL = None      # e.g. "id"/"name"; None -> use dataframe index
TOP_N = 3                    # nearest PDB ligands to report per compound
# ----------------

mfp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def prep(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None: return None
    return dict(mol=m,
                can=Chem.MolToSmiles(m),
                ik=MolToInchiKey(m),
                ik14=MolToInchiKey(m).split("-")[0],           # connectivity layer only
                scaf=Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(m)),
                fp=mfp.GetFingerprint(m))

# --- build the PDB ligand reference set (one row per unique ligand) ---
summ = pd.read_csv(SUMMARY_CSV)
pdb_ligs = {}
for _, r in summ.iterrows():
    codes  = str(r.get("ligand_codes",  "") or "").split(";")
    smis   = str(r.get("ligand_smiles", "") or "").split(" | ")
    names  = str(r.get("ligand_names",  "") or "").split(" | ")
    for code, smi, nm in zip(codes, smis, names):
        code = code.strip()
        if not code or smi.strip() in ("", "nan"): continue
        p = prep(smi)
        if p is None: continue
        if code not in pdb_ligs:
            pdb_ligs[code] = dict(code=code, name=nm.strip(), entries=set(), **p)
        pdb_ligs[code]["entries"].add(r["pdb_id"])
print(f"{len(pdb_ligs)} unique drug-like ligands across {len(summ)} PDB entries")

ref = list(pdb_ligs.values())
ref_fps = [x["fp"] for x in ref]

# --- compare each challenge compound ---
rows = []
for idx, r in test_structure.iterrows():
    cid = str(r[CHALLENGE_ID_COL]) if CHALLENGE_ID_COL else f"cmpd_{idx}"
    q = prep(r[CHALLENGE_SMILES_COL])
    if q is None:
        rows.append(dict(compound=cid, smiles=r[CHALLENGE_SMILES_COL], note="UNPARSEABLE")); continue

    sims = np.array(DataStructs.BulkTanimotoSimilarity(q["fp"], ref_fps))
    order = np.argsort(-sims)[:TOP_N]

    exact      = [x for x in ref if x["ik"]   == q["ik"]]
    same_conn  = [x for x in ref if x["ik14"] == q["ik14"]]      # ignores stereo/charge/tautomer
    same_scaf  = [x for x in ref if x["scaf"] == q["scaf"] and q["scaf"]]

    best = ref[order[0]]
    rows.append(dict(
        compound=cid,
        smiles=q["can"],
        exact_match      = ";".join(sorted(x["code"] for x in exact)),
        same_connectivity= ";".join(sorted(x["code"] for x in same_conn)),
        same_scaffold    = ";".join(sorted(x["code"] for x in same_scaf)),
        nearest_code     = best["code"],
        nearest_name     = best["name"][:60],
        nearest_tanimoto = round(float(sims[order[0]]), 3),
        nearest_pdb_ids  = ";".join(sorted(best["entries"])[:5]),
        top_hits         = "; ".join(f"{ref[i]['code']}:{sims[i]:.2f}" for i in order),
        risk = ("EXACT" if exact else
                "SAME-CONNECTIVITY" if same_conn else
                "SCAFFOLD" if same_scaf else
                "HIGH-SIM" if sims[order[0]] >= 0.7 else
                "MODERATE" if sims[order[0]] >= 0.4 else "NOVEL"),
    ))

xref = pd.DataFrame(rows).sort_values("nearest_tanimoto", ascending=False)
xref = xref.sort_values(
    by="compound", key=lambda s: s.str.extract(r"(\d+)").astype(int)[0]
).reset_index(drop=True)
xref.to_csv("cyp3a4_challenge_vs_pdb.csv", index=False)

print("\nrisk breakdown:", xref["risk"].value_counts().to_dict())
print()
display(xref[["compound","risk","nearest_code","nearest_name","nearest_tanimoto",
              "exact_match","same_scaffold","nearest_pdb_ids"]])

## 3. Exploratory data analysis (taken directly from tutorial code)

Before moving to structure prediction, we first inspect the blinded ligand set. This helps us understand whether the ligands have reasonable diversity and physicochemical ranges for a CYP3A4-focused structure task.

In this section we will:

1. Visualize a sample of ligand structures with RDKit
2. Estimate scaffold diversity using Murcko scaffolds
3. Plot the top recurring scaffolds
4. Summarize key descriptor distributions (MW, LogP, TPSA, heavy atoms)

In [ ]:
eda_df = structure_df.dropna(subset=["mol"]).copy()
print(f"Valid molecules: {len(eda_df)} / {len(structure_df)}")

if eda_df.empty:
    raise ValueError("No valid molecules available. Check SMILES parsing in step 2.")

sample_n = min(24, len(eda_df))
sample_mols = eda_df["mol"].head(sample_n).tolist()
sample_legends = eda_df["id"].head(sample_n).tolist()

img = Draw.MolsToGridImage(
    sample_mols,
    molsPerRow=6,
    subImgSize=(220, 180),
    legends=sample_legends,
)
img

To analyze structural diversity, we compute **Bemis-Murcko scaffolds** for each ligand and count unique scaffold chemotypes.

In [ ]:
eda_df["murcko_mol"] = eda_df["mol"].apply(
    lambda molecule: MurckoScaffold.GetScaffoldForMol(molecule) if molecule is not None else None
)
eda_df["murcko_smiles"] = eda_df["murcko_mol"].apply(
    lambda scaffold: Chem.MolToSmiles(scaffold) if scaffold is not None else ""
)

# Keep non-empty scaffolds for counting
murcko_series = eda_df.loc[eda_df["murcko_smiles"] != "", "murcko_smiles"]
num_unique_scaffolds = murcko_series.nunique()
print(f"Number of unique Murcko scaffolds: {num_unique_scaffolds}")

Now we visualize the frequency of the scaffolds. The bar chart shows scaffold counts, and scaffold images are overlaid below each bar for quick structural interpretation. Only two scaffold are repeated twice, so the set is reasonably diverse.

In [ ]:
def add_scaffold_images(axis, smiles_list, image_size=(120, 120), zoom=0.5, y_offset=-60):
    for index, smiles in enumerate(smiles_list):
        if not smiles:
            continue

        molecule = Chem.MolFromSmiles(smiles)
        if molecule is None:
            continue

        scaffold_image = Draw.MolToImage(molecule, size=image_size)
        image_box = OffsetImage(scaffold_image, zoom=zoom)
        annotation = AnnotationBbox(
            image_box,
            (index, 0),
            xybox=(0, y_offset),
            xycoords=("data", "data"),
            boxcoords="offset points",
            frameon=False,
        )
        axis.add_artist(annotation)

In [ ]:
scaffold_counts = murcko_series.value_counts()
scaffolds_df = scaffold_counts.rename_axis("smiles").reset_index(name="count")

fig, axis = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=scaffolds_df,
    x=scaffolds_df.index,
    y="count",
    color="steelblue",
    ax=axis,
)

add_scaffold_images(axis, scaffolds_df["smiles"])

axis.set_ylabel("Count")
axis.set_xlabel("Murcko Scaffolds")
axis.set_title(f"Frequency of Scaffolds (N molecules = {len(eda_df)})")
axis.set_xticklabels([])

axis.set_ylim(0, scaffolds_df["count"].max() * 1.25)
axis.grid(axis="y", linestyle="--", alpha=0.6)
fig.subplots_adjust(bottom=0.22)
plt.tight_layout()

We also summarize key physicochemical properties with RDKit descriptors to check whether the ligand set occupies a reasonable range for structure-based modeling.

In [ ]:
eda_df["MW"] = eda_df["mol"].apply(Descriptors.MolWt)
eda_df["LogP"] = eda_df["mol"].apply(Descriptors.MolLogP)
eda_df["TPSA"] = eda_df["mol"].apply(rdMolDescriptors.CalcTPSA)
eda_df["HeavyAtoms"] = eda_df["mol"].apply(lambda molecule: molecule.GetNumHeavyAtoms())

descriptor_specs = [
    ("MW", "Molecular Weight (Da)"),
    ("LogP", "LogP"),
    ("TPSA", "TPSA (Å²)"),
    ("HeavyAtoms", "Heavy Atom Count"),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
for axis, (column, xlabel) in zip(axes.flatten(), descriptor_specs):
    sns.histplot(eda_df[column], bins=30, kde=True, color="steelblue", ax=axis)
    axis.set_title(f"{column} (N = {eda_df[column].notna().sum()})")
    axis.set_xlabel(xlabel)

plt.tight_layout()

## 6. Prepare submission and validate

The following are the submission requirements for this track:

- Upload a `.zip` archive containing one PDB file per ligand in the blinded structure set.
- Name each file `{id}.pdb` (e.g., matching the `id` column in `structure_df`).
- In each file, the ligand residue name must be exactly **`LIG`**.



In [ ]:
# This will harvest all of the poses into a single folder as well as score the best
# pose based on the iptm_AC and iptm_BC scores (first is protein:ligand, second is ligand:heme)
# The two are tightly correlated ligand:heme is used as a tie breaker when protein:ligand scores are identical



load_dotenv(find_dotenv(), override=True)

# --- CONFIGURATION ---
RUN_NAME = "openadmet20_run2"
HARVEST_INPUT_DIR = Path(os.getenv("HARVEST_INPUT_DIR"))
OUTPUT_DIR  = HARVEST_INPUT_DIR / "harvested_top"          # best pose per compound
ALL_DIR     = HARVEST_INPUT_DIR / "all_predictions"    # every pose, flat, for visual inspection
SUMMARY_CSV = HARVEST_INPUT_DIR / f"harvest_summary_{RUN_NAME}.csv"
ALL_SCORES_CSV = HARVEST_INPUT_DIR / f"harvest_all_scores_{RUN_NAME}.csv"   # one row per pose
SKIP_DIRS   = {"logs", OUTPUT_DIR.name, ALL_DIR.name}   # <-- don't scan our own outputs

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ALL_DIR.mkdir(parents=True, exist_ok=True)



harvested_data = []
all_scores = [] 

print(f"Scanning {HARVEST_INPUT_DIR} for the best prediction per target...")

# Iterate over every subfolder in the submission directory
for target_folder in HARVEST_INPUT_DIR.iterdir():
    if not target_folder.is_dir() or target_folder.name in SKIP_DIRS:
        continue
        
    target_id = target_folder.name
    best_cif_path = None
    best_score = (-float('inf'), -float('inf'))
    best_model_name = ""
    
    # Use rglob to search inside 'seed_42' or any other subfolders
    structure_files = list(target_folder.rglob("*.cif")) + list(target_folder.rglob("*.pdb"))
    
    if not structure_files:
        print(f"  [!] No structures found in {target_id}")
        continue

    for struct_path in structure_files:
        cif_stem = struct_path.stem
        score = -float('inf')
        
        # Remove the '_model' suffix from the .cif stem to get the base sample name
        # e.g., 'x00370-1_seed_42_sample_1_model' -> 'x00370-1_seed_42_sample_1'
        base_name = cif_stem.replace("_model", "")
        print (base_name)
        
        # Look for the exact OpenFold3 JSON naming conventions
        parent_dir = struct_path.parent
        possible_jsons = [
            parent_dir / f"{base_name}_confidences_aggregated.json",
            #parent_dir / f"{base_name}_confidences.json"
        ]
        
        json_found = False
        for json_path in possible_jsons:
            if json_path.exists():
                try:
                    with open(json_path, 'r') as f:
                        data = json.load(f)
                        
                    # Extract the ranking score, falling back to iptm, then ptm
                    #score = data.get("iptm",
                    #        data.get("ranking_score", 
                    #        data.get("ptm", -float('inf'))))
                    cp = data.get("chain_pair_iptm", {})
                    ac = cp.get("(A, C)", data.get("iptm", -float('inf')))   # protein-ligand
                    bc = cp.get("(B, C)", -float('inf'))                      # heme-ligand tiebreak

                    if data.get("has_clash", 0):
                        score = (-float('inf'), -float('inf'))   # disqualify
                    else:
                        score = (ac, bc)                    
                    json_found = True
                    break
                except Exception:
                    continue # If JSON fails to parse, try the next possible file
        
        shutil.copy2(struct_path, ALL_DIR / f"{target_id}__{cif_stem}{struct_path.suffix}")

        if not json_found or score[0] == -float('inf'):
            print(f"  [?] No valid score for {cif_stem} in {target_id}")
            continue

        all_scores.append({
            "Target_ID": target_id,
            "Model": cif_stem,
            "seed": struct_path.parent.name,          # e.g. "seed_42"
            "iptm_AC": round(score[0], 4),
            "iptm_BC": round(score[1], 4),
        })

        # # copy every scored pose into the flat inspection folder
        # shutil.copy2(struct_path, ALL_DIR / f"{target_id}__{cif_stem}_s{score:.3f}{struct_path.suffix}")

            
        # Update the champion if this model beats the current high score
        if score > best_score:
            best_score = score
            best_cif_path = struct_path
            best_model_name = cif_stem

    # Once all samples are checked, copy the champion to the final folder
    if best_cif_path and best_score != -float('inf'):
        final_ext = best_cif_path.suffix
        final_file_path = OUTPUT_DIR / f"{target_id}{final_ext}"
        
        shutil.copy2(best_cif_path, final_file_path)
        
        harvested_data.append({
            "Target_ID": target_id,
            "Winning_Model": best_model_name,
            "iptm_AC": round(best_score[0], 4),
            "iptm_BC": round(best_score[1], 4),
            "File_Name": final_file_path.name
        })
        print(f"  [+] {target_id}: {best_model_name} (A-C {best_score[0]:.3f}, B-C {best_score[1]:.3f})")
    else:
        print(f"  [!] Failed to extract best score for {target_id}")

# Compile the harvest into a Pandas DataFrame and export
if harvested_data:
    df_harvest = pd.DataFrame(harvested_data)
    # Sort by Score descending to see the best predictions at the top
    df_harvest = df_harvest.sort_values(
        by=["iptm_AC", "iptm_BC"],
        ascending=[False, False]
    )
    df_harvest.to_csv(SUMMARY_CSV, index=False)
    
    print("\n" + "="*50)
    print(f" HARVEST COMPLETE")
    print("="*50)
    print(f" Successfully processed {len(df_harvest)} targets.")
    print(f" Copied best structures to: {OUTPUT_DIR}/")
    print(f" Copied all structures to: {ALL_DIR}/")
    print(f" Saved metrics to: {SUMMARY_CSV}")
if all_scores:
    pd.DataFrame(all_scores).to_csv(ALL_SCORES_CSV, index=False)
    print(f" Saved {len(all_scores)} per-pose scores to: {ALL_SCORES_CSV}")
else:
    print("\n[!] No valid files were harvested. Check the directory structure or JSON formatting.")

In [ ]:
# Visualize per-pose interface confidence across seeds for each compound

df = pd.read_csv(ALL_SCORES_CSV)
order = sorted(
    df["Target_ID"].unique(),
    key=lambda target_id: int(target_id.removeprefix("cmpd_"))
)
grouped = [df.loc[df["Target_ID"] == t, "iptm_AC"].values for t in order]


fig, ax = plt.subplots(figsize=(max(10, len(order) * 0.6), 6))
ax.boxplot(grouped, labels=order, whis=1.5, showfliers=False)

rng = np.random.default_rng(42)
for i, t in enumerate(order, start=1):
    sub = df[df["Target_ID"] == t]
    x = i + rng.uniform(-0.18, 0.18, size=len(sub))
    ax.scatter(x, sub["iptm_AC"], s=5, alpha=0.5, c="tab:blue", linewidths=0, zorder=3,
               label="iptm A-C (protein-ligand)" if i == 1 else None)
    ax.scatter(x, sub["iptm_BC"], s=5, alpha=0.5, c="tab:orange", linewidths=0, zorder=3,
               label="iptm B-C (heme-ligand)" if i == 1 else None)

ax.set_xlabel("Compound", fontsize=14)
ax.set_ylabel("chain-pair ipTM", fontsize=14)
ax.set_title(f"Per-pose interface confidence across seeds — {RUN_NAME}", fontsize=16)
ax.legend()
plt.xticks(rotation=90)
plt.tight_layout(); plt.show()

# spread: where does pose selection matter most?
spread = (df.groupby("Target_ID")["iptm_AC"]
            .agg(["count", "min", "median", "max"])
            .assign(range=lambda d: d["max"] - d["min"])
            .sort_values("range", ascending=False))
print(spread.to_string())

 We use this helper to convert our  predictions  into submission-ready PDBs. Adjust `cif_dir` to point to your own prediction directory here.
 

In [ ]:
load_dotenv(find_dotenv(), override=True)
cif_dir = os.getenv("cif_dir")
#cif_dir = OUTPUT_DIR
submission_output_dir = os.getenv("submission_output_dir")
print(f"cif_dir: {cif_dir}")
print(f"submission_output_dir: {submission_output_dir}")

submission_output_dir = Path(submission_output_dir)
cif_dir = Path(cif_dir)
submission_output_dir.mkdir(parents=True, exist_ok=True)

for cif_file in sorted(cif_dir.glob("cmpd_*.cif")):
    print(f"Processing {cif_file.name}")
    # We extract the ligand ID from the filename, e.g. cyp3a4_ocnt-2312554_model_0.cif -> ocnt-2312554.
    # In this example we choose the "model_0" pose, wich is the top-ranked pose according to Boltz-2's confidence scoring.
    # Also, we need to rename the output files to match the submission naming convention, which requires the OCNT id in uppercase.
    #ligand_id = cif_file.stem.removeprefix("cyp3a4_").removesuffix("_model_0").upper()
    ligand_num = cif_file.stem.removeprefix("cmpd_").upper()
    ligand_num = int(ligand_num)
    ligand_id = structure_df["id"].iloc[ligand_num]
    out_pdb = submission_output_dir / f"{ligand_id}.pdb"
    try:
        rewrite_ligand_resname_pdb(cif_file, out_pdb, old_resname="LIG0", new_resname="LIG")
        print(f"Converted {cif_file.name} -> {out_pdb.name}")
    except Exception as exc:
        print(f"Failed {cif_file.name}: {exc}")

### Validate & prepare zipfile for upload



In [ ]:
# PoseBusters analysis for predicted CYP3A4 structures 
# (optional - OpenADMET will run on submission and failed structures won't be scored)
# PoseBusters here is oversensitive to overlap and always shows protein:ligand clashes
# The OpenADMET Posebusters seems to be a bit more tolerant to minor clashes

RDLogger.DisableLog("rdApp.*")
load_dotenv(find_dotenv(), override=True)

# ---- config ----
SUBMISSION_DIR = os.getenv("SUBMISSION_DIR")
CHALLENGE_CSV = os.getenv("CHALLENGE_CSV")  # id, smiles
EXTRACT_DIR = os.getenv("EXTRACT_DIR")
OUT_CSV = os.getenv("OUT_CSV")
ID_COL, SMILES_COL = "id", "SMILES"
PROTEIN_CHAIN  = "A"
LIGAND_RESNAME = "LIG"      # after your LIG0 -> LIG rename
# ----------------
print(f"EXTRACT_DIR: {EXTRACT_DIR}")
print(f"SUBMISSION_DIR: {SUBMISSION_DIR}")
print(f"CHALLENGE_CSV: {CHALLENGE_CSV}")
print(f"OUT_CSV: {OUT_CSV}")
os.makedirs(EXTRACT_DIR, exist_ok=True)

# --- reference 2D mols from challenge SMILES, keyed by submission filename stem ---
df_master = pd.read_csv(CHALLENGE_CSV)
lm_dict = {}
for _, row in df_master.iterrows():
    smi = row[SMILES_COL]
    if pd.notna(smi):
        m = Chem.MolFromSmiles(str(smi))
        if m:
            # submission files are named e.g. "0.pdb" from cmpd_0 -> uppercase stem
            key = str(row[ID_COL]).strip().removeprefix("cmpd_").upper()
            lm_dict[key] = Chem.AddHs(m)
print(f"{len(lm_dict)} reference molecules: {sorted(lm_dict)[:5]} ...")

pb = PoseBusters(config="dock")
pdb_files = sorted(glob.glob(os.path.join(SUBMISSION_DIR, "*.pdb")))
print(f"{len(pdb_files)} submission PDBs")

pb_results = []
for pdb_path in tqdm(pdb_files, desc="PoseBusters"):
    clean_id = Path(pdb_path).stem
    if clean_id not in lm_dict:
        print(f"  [!] no reference SMILES for {clean_id}; skipping")
        continue
    ref_mol = lm_dict[clean_id]
    protein_pdb = os.path.join(EXTRACT_DIR, f"{clean_id}_protein.pdb")

    try:
        # 1. isolate protein chain (heme/ligand chains removed)
        st = gemmi.read_structure(pdb_path)
        prot = st.clone()
        for model in prot:
            for cname in [c.name for c in model if c.name != PROTEIN_CHAIN]:
                model.remove_chain(cname)
        prot.write_pdb(protein_pdb)

        # 2. extract the compound by RESNAME (not chain) so heme can't be picked up
        mol_complex = Chem.MolFromPDBFile(pdb_path, sanitize=False, removeHs=False)
        lig_atoms = [a.GetIdx() for a in mol_complex.GetAtoms()
                     if a.GetMonomerInfo()
                     and a.GetMonomerInfo().GetResidueName().strip() == LIGAND_RESNAME]
        if not lig_atoms:
            raise ValueError(f"no residue named {LIGAND_RESNAME} found")
        pred_raw = Chem.PathToSubmol(mol_complex, lig_atoms)

        # 3. graft correct bond orders from the SMILES template onto the 3D coords
        try:
            pred_fixed = AllChem.AssignBondOrdersFromTemplate(ref_mol, pred_raw)
        except ValueError:
            pred_fixed = pred_raw
            for b in pred_fixed.GetBonds():
                b.SetBondType(Chem.rdchem.BondType.SINGLE)

        # 4. bust
        result = pb.bust(mol_pred=pred_fixed, mol_true=None,
                         mol_cond=protein_pdb, full_report=False)

        ignore = ["mol_pred", "mol_true", "mol_cond", "mol_true_loaded"]
        test_cols = [c for c in result.columns if c not in ignore]
        row = {"compound": clean_id, "pb_error": None}
        n_pass = 0
        for col in test_cols:
            passed = bool(result[col].iloc[0])
            row[f"pb_{col.replace('pb_','')}"] = passed
            n_pass += int(passed)
        row.update(pb_pass=(n_pass == len(test_cols)), pb_n_pass=n_pass, pb_n_total=len(test_cols))
        pb_results.append(row)

    except Exception as e:
        pb_results.append({"compound": clean_id, "pb_pass": False,
                           "pb_n_pass": 0, "pb_n_total": 0, "pb_error": str(e)})

pb_df = pd.DataFrame(pb_results)
pb_df.to_csv(OUT_CSV, index=False)

print(f"\n=== PoseBusters Summary ===")
print(f"Total:            {len(pb_df)}")
print(f"All checks pass:  {pb_df['pb_pass'].sum()} ({pb_df['pb_pass'].mean()*100:.1f}%)")
print(f"One or more fail: {(~pb_df['pb_pass']).sum()}")
if pb_df["pb_error"].notna().any():
    print("\nerrors:")
    for _, r in pb_df[pb_df["pb_error"].notna()].iterrows():
        print(f"  {r['compound']}: {r['pb_error']}")

check_cols = [c for c in pb_df.columns if c.startswith("pb_")
              and c not in ("pb_pass","pb_n_pass","pb_n_total","pb_error")]
fails = pb_df[check_cols].apply(lambda c: (c == False).sum())
fails = (fails[fails > 0] / len(pb_df)).sort_values(ascending=False)
print(f"\nMost common failures:\n{fails.to_string() if len(fails) else '  none'}")
print(f"\n-> {OUT_CSV}")

In [ ]:
# PDB files  in `output_dir` (one `{id}.pdb` per ligand)

# This OpenADMET validation code also checks that your submitted structures have the correct bond orders 
# (corresponding to the SMILES given) and that each complex has at most three chains — protein, heme cofactor, and ligand. 
# The validation function takes an `id: SMILES` mapping, which we build directly from `structure_df`.



load_dotenv(find_dotenv(), override=True)

# Find the project root before importing the validation module
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

#from validation.structure_validation import validate_structure_submission
from validation import validate_structure_submission

# Point this to your directory of predicted PDB files

processed_submission_dir = Path(os.getenv("processed_submission_dir"))
zip_path = Path(os.getenv("zip_path"))



# Build an ID -> SMILES mapping for ligand chemistry validation (connectivity and bond orders).
expected_ligand_smiles = dict(zip(structure_df["id"], structure_df["SMILES"]))


pdb_files = sorted(processed_submission_dir.glob("*.pdb"))
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for pdb in pdb_files:
        zf.write(pdb, arcname=pdb.name)

print(f"Created zip archive: {zip_path} ({len(pdb_files)} files)")
print(f"Built SMILES map for {len(expected_ligand_smiles)} ligands")

# Suppress noisy warning logs from RDKit and MDAnalysis during validation.
RDLogger.DisableLog("rdApp.warning")
try:
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning, module=r"MDAnalysis.*")
        warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"MDAnalysis.*")

        is_valid, validation_errors = validate_structure_submission(
            zip_path,
            expected_ids=set(structure_df["id"]),
            expected_ligand_smiles=expected_ligand_smiles,
            require_lig_resname=True,
            check_numbering=True,
        )
finally:
    RDLogger.EnableLog("rdApp.warning")

if is_valid:
    print("✅ Submission is valid — ready to upload.")
else:
    print("❌ Submission is invalid:")
    for msg in validation_errors:
        print(f"  - {msg}")

In [ ]:
# Read the first PDB file and print chain and residue information - all others should be the same (spot check by changing from 0)
st = gemmi.read_structure(pdb_files[0])
for model in st:
    for ch in model:
        print(ch.name, {r.name for r in ch})

## Submission via API - taken directly from tutorial

Go to the web portal and submit there, carefully filling in your information.

https://huggingface.co/spaces/openadmet/cyp-challenge

In [ ]:

load_dotenv(find_dotenv(), override=True)
# get a token to submit
hf_token = get_token()
client = Client("openadmet/cyp-challenge", token=hf_token)

# Replace this with the path to your submission zip file
#file_path = Path("/home/byron/projects/openadmet/cyp/openfold_work/cyp_predictions/openadmet20_run1/cyp3a4_structure_submission_run1C.zip")
#file_path
# Replace this with your own information and uncomment to submit.
result = client.predict(
   username= os.getenv("username"),
   user_alias=os.getenv("user_alias"), # Replace if anonymous
   anon_checkbox=True, # Change to True if you want to submit anonymously
   participant_name=os.getenv("participant_name"),
   discord_username=os.getenv("discord_username"),
   email=os.getenv("email"),
   affiliation=os.getenv("affiliation"),
   model_tag=os.getenv("model_tag"),
   paper_checkbox=True,
   proprietary_data_checkbox=False,
   open_code_checkbox=True,
   track_select="Structure Prediction",
   file_input=handle_file(os.getenv("zip_path")),
   api_name="/submit_predictions",
)

print(result)

## UNUSED CELLS FROM EXAMPLE

## 4. Generating a protein-ligand structure

For this tutorial, we avoid running a heavy structure-prediction model and instead show how to build the input for a **Boltz-2** co-folding run.

To generate a protein-ligand structure with Boltz-2, you need a `YAML` input file containing the protein sequence and ligand SMILES. In the cells below, we create that YAML directly from:

- the local CYP3A4 FASTA sequence
- the SMILES of a specific ligand from the blinded structure set

In [ ]:
boltz_yaml = Path("../inputs/cyp3a4_example.yaml")

example_ligand_id = structure_df["id"].iloc[0]
example_rows = structure_df.loc[structure_df["id"] == example_ligand_id, ["id", "SMILES"]]
if example_rows.empty:
    raise ValueError(f"Could not find ligand '{example_ligand_id}' in structure_df")

example_smiles = example_rows.iloc[0]["SMILES"]
example_yaml = f"""version: 1
sequences:
- protein: {{id: A, sequence: {cyp3a4_sequence}}}
- ligand: {{id: B, smiles: {example_smiles}}}
properties:
- affinity: {{binder: B}}
"""

boltz_yaml.write_text(example_yaml)
print(f"Wrote YAML file: {boltz_yaml}")
print("\nYAML preview:")
print("-" * 80)
print(example_yaml)

## 5. Comparing prediction with reference

When reference crystal structures are available, each predicted complex is scored against its crystallographic target using OpenStructure (OST):

- **SCRMSDScorer**: Superimposes the binding site (Cα atoms within 8 Å of the reference ligand), computes symmetry-corrected ligand RMSD, and also reports LDDT-LP.
- **LDDTPLIScorer**: Computes LDDT-PLI without requiring superposition, evaluating whether protein-ligand contacts are preserved.


### Load the CYP3A4 protein sequence from a FASTA file

This FASTA contains a **monomeric** CYP3A4 sequence, which was extracted from [UniProt](https://www.uniprot.org/uniprotkb/P08684/entry#sequences). No additional structure information should be needed for the challenge submission.

In [ ]:
fasta_path = Path("../inputs/CYP3A4_protein_sequence.fasta")

if not fasta_path.exists():
    raise FileNotFoundError(f"Missing FASTA file: {fasta_path}")

fasta_text = fasta_path.read_text().strip()
fasta_lines = fasta_text.splitlines()
fasta_header = fasta_lines[0]
cyp3a4_sequence = "".join(line.strip() for line in fasta_lines[1:])

print(fasta_header)
print(f"Sequence length: {len(cyp3a4_sequence)} aa")
display(fasta_text)

Now that we've converted our predictions to submission-ready PDBs, we can preview one of the resulting complexes using NGLView. This cell is optional and requires the `nglview` package.

In [ ]:
converted_pdbs = sorted(output_dir.glob("*.pdb"))

if nv is not None and converted_pdbs:
    boltz_pdb = converted_pdbs[0]
    v = nv.show_file(str(boltz_pdb))
    v.clear_representations()
    v.add_representation("cartoon", selection="protein", color="lightgreen")
    v.add_representation("ball+stick", selection="hetero")
    display(v)
elif nv is None:
    print("nglview is not installed — skipping structure visualisation. Install with: pip install nglview")
else:
    print(f"No converted PDB files found in {output_dir} yet — run the conversion cell above first.")

In [ ]:
# dev distance inspection
from rdkit import Chem
import numpy as np, gemmi

SUBMISSION_DIR = Path("/home/byron/projects/openadmet/cyp/openfold_work/cyp_predictions/openadmet20_run1/my_structure_submission")
OUT_DIR = Path("/home/byron/projects/openadmet/cyp/openfold_work/cyp_predictions/openadmet20_run1/minimized_submission")

def lig_mol_from_pdb(path, resname="LIG"):
    st = gemmi.read_structure(str(path)); st.setup_entities()
    xyz, els = [], []
    for ch in st[0]:
        for r in ch:
            if r.name.strip() == resname:
                for a in r:
                    if a.element.name == "H": continue
                    xyz.append([a.pos.x, a.pos.y, a.pos.z]); els.append(a.element.name)
    return np.array(xyz), els

for stem in ["OCNT-2313038", "OCNT-2313300"]:
    for label, d in (("orig", SUBMISSION_DIR), ("min", OUT_DIR)):
        xyz, els = lig_mol_from_pdb(d / f"{stem}.pdb")
        # longest and shortest interatomic distances that look like bonds
        n = len(xyz)
        ds = [(np.linalg.norm(xyz[i]-xyz[j]), els[i], els[j], i, j)
              for i in range(n) for j in range(i+1, n)]
        bonds = sorted([b for b in ds if b[0] < 2.2])
        print(f"{stem} {label}: {n} heavy atoms | "
              f"shortest {bonds[0][0]:.2f} ({bonds[0][1]}-{bonds[0][2]}) | "
              f"longest {bonds[-1][0]:.2f} ({bonds[-1][1]}-{bonds[-1][2]})")

In [ ]:
# dev distance inspection
from rdkit import Chem
import tempfile, os

def perceived(path, resname="LIG"):
    st = gemmi.read_structure(str(path)); st.setup_entities()
    lines, n = [], 0
    for ch in st[0]:
        for r in ch:
            if r.name.strip() != resname: continue
            for a in r:
                if a.element.name == "H": continue
                n += 1
                lines.append(f"HETATM{n:>5} {a.name:<4} LIG A   1    "
                             f"{a.pos.x:8.3f}{a.pos.y:8.3f}{a.pos.z:8.3f}  1.00  0.00"
                             f"          {a.element.name.upper():>2}\n")
    tmp = tempfile.NamedTemporaryFile(suffix=".pdb", delete=False, mode="w")
    tmp.writelines(lines); tmp.write("END\n"); tmp.close()
    m = Chem.MolFromPDBFile(tmp.name, removeHs=True, sanitize=False)
    os.unlink(tmp.name)
    bonds = {tuple(sorted((b.GetBeginAtomIdx(), b.GetEndAtomIdx()))) for b in m.GetBonds()}
    return m, bonds

for stem in ["OCNT-2313038", "OCNT-2313300"]:
    m0, b0 = perceived(Path(SUBMISSION_DIR) / f"{stem}.pdb")
    m1, b1 = perceived(Path(OUT_DIR) / f"{stem}.pdb")
    print(f"{stem}: orig {len(b0)} bonds, min {len(b1)} bonds")
    for i, j in sorted(b0 - b1):
        print(f"   LOST: {m0.GetAtomWithIdx(i).GetSymbol()}{i}-{m0.GetAtomWithIdx(j).GetSymbol()}{j}")
    for i, j in sorted(b1 - b0):
        print(f"   GAINED: {m1.GetAtomWithIdx(i).GetSymbol()}{i}-{m1.GetAtomWithIdx(j).GetSymbol()}{j}")